# 🧬 Byte-Level BPE Tokenizer (GPT-3 Style)

Unlike traditional tokenizers that operate on words or characters, Byte-Level BPE tokenizers work directly on raw bytes (UTF-8 encoded). This enables them to handle **all scripts, emojis, and special symbols** without needing separate preprocessing.

Used in: **GPT-3**, **GPT-Neo**, **Codex**, **OpenAI Tokenizer**

---

## Why Byte-Level?

🧠 Models must handle:
- Emojis (😊, ❤️)
- Multilingual text (こんにちは, مرحبا)
- Typos, rare words, and special characters

➡️ Byte-level tokenization ensures that **every input** can be encoded without needing a fixed vocabulary of characters.


In [1]:
# Step 1 : Define a small corpus
corpus = [
    "Hello World",
    "Hello 😊",
    "Hello darkness my old friend",
]

# Step 2 : Convert each sentence to a list of UTF-8 bytes
def get_byte_corpus(corpus) :
    return [list(sentence.encode('utf-8')) for sentence in corpus]

byte_corpus =get_byte_corpus(corpus)

# Print each line's byte representation
for i, line in enumerate(byte_corpus) :
    print(f"Line {i+1} bytes :", line)

Line 1 bytes : [72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]
Line 2 bytes : [72, 101, 108, 108, 111, 32, 240, 159, 152, 138]
Line 3 bytes : [72, 101, 108, 108, 111, 32, 100, 97, 114, 107, 110, 101, 115, 115, 32, 109, 121, 32, 111, 108, 100, 32, 102, 114, 105, 101, 110, 100]


### 🔹 Step 1: Encode the Corpus

We start by converting each sentence into its **raw byte values** using UTF-8. This ensures full support for multilingual characters and emojis.


In [11]:
from collections import Counter

def build_vocab(byte_corpus):
    vocab = Counter()
    for line in byte_corpus:
        # Make sure everything is `bytes`, not integers or lists
        tokens = [b if isinstance(b, bytes) else bytes([b]) for b in line]

        for i in range(len(tokens) - 1):
            vocab[tokens[i] + tokens[i+1]] += 1
    return vocab


# Build vocab from byte-level corpus
vocab = build_vocab(byte_corpus)

# Print top 5 most frequent byte pairs
print("Sample byte pair frequencies : ", list(vocab.items())[:5])

Sample byte pair frequencies :  [(b'He', 3), (b'el', 3), (b'll', 3), (b'lo', 3), (b'o ', 3)]


### 🔹 Step 2: Build Vocabulary of Byte Pairs

Here, we scan the byte sequences and count all **frequent adjacent byte pairs**. These are the candidates to be merged using BPE.


In [12]:
#  Helper to flatten  nested tokens
# ✅ Helper: Flatten nested byte tokens
def flatten(seq):
    result = []
    for item in seq:
        if isinstance(item, (list, tuple)):
            result.extend(item)
        else:
            result.append(item)
    return result

def merge_vocab(byte_corpus, num_merges=5):
    merges = []

    for _ in range(num_merges):
        pair_freq = build_vocab(byte_corpus)
        if not pair_freq:
            break

        best_pair = max(pair_freq, key=pair_freq.get)
        merges.append(best_pair)

        new_corpus = []
        for line in byte_corpus:
            i = 0
            new_line = []
            while i < len(line):
                a = line[i]
                b = line[i+1] if i+1 < len(line) else None

                # Convert to bytes if not already
                a = a if isinstance(a, bytes) else bytes([a])
                b = b if isinstance(b, bytes) else bytes([b]) if b is not None else None

                if b and a + b == best_pair:
                    new_line.append(a + b)
                    i += 2
                else:
                    new_line.append(a)
                    i += 1

            new_corpus.append(new_line)

        byte_corpus = [flatten(line) for line in new_corpus]

    return merges



# Perform 5 merge operations
byte_merges = merge_vocab(byte_corpus, num_merges=5)
print(f'Learned byte merges : {byte_merges}')
    

Learned byte merges : [b'He', b'Hel', b'Hell', b'Hello', b'Hello ']


### 🔹 Step 3: Merge Most Frequent Byte Pairs

We apply BPE-style merging by replacing the most frequent pairs in the corpus with single units. Each merge creates a new token.


In [13]:
# Step 5 : Tokenize a new setence using learned merges
def byte_tokenize(text, merges) :
    input_bytes = list(text.encode("utf-8"))
    tokens = [bytes([b]) for b in input_bytes]

    for merge in merges :
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] + tokens[i+1] == merge :
                tokens[i] = tokens[i] + tokens[i+1]
                del tokens[i+1]
            else :
                i += 1


    return tokens

# Test the tokenizer on new text
test_text = "Hello 🌍"
tokens = byte_tokenize(test_text, byte_merges)
print(f"Tokenized output  : {tokens}")

Tokenized output  : [b'Hello ', b'\xf0', b'\x9f', b'\x8c', b'\x8d']


### 🔹 Step 4: Tokenize New Sentences

Using the learned byte-level merges, we tokenize new inputs by applying the same merge rules. Even emojis and symbols are handled seamlessly.


## ✅ Summary: Byte-Level BPE

Byte-Level BPE is powerful and flexible, especially for multilingual and low-resource text.

### ✅ Pros:
- Supports **any UTF-8 character** (emoji, Chinese, Arabic, etc.)
- No unknown tokens
- Works directly on raw bytes

### ❌ Cons:
- Longer token sequences
- Greedy and fixed merging
- More memory during pretraining

---